In [ ]:
# ==========================================
# CELL 1: IMPORT THƯ VIỆN & TẢI DỮ LIỆU THÔ
# ==========================================
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

print("⏳ Đang tải dữ liệu từ file CSV...")
df_ml = pd.read_csv('dashboard_data.csv')

# Lấy mẫu ngẫu nhiên 300,000 dòng để 16GB RAM chạy mượt mà
df_sample = df_ml.sample(n=300000, random_state=42).copy()
print(f"✅ Đã tải xong! Kích thước dữ liệu ban đầu: {df_sample.shape}")

In [ ]:
# ==========================================
# CELL 2: DATA CLEANING & ANTI-LEAKAGE
# ==========================================
print("🧹 Đang dọn dẹp các cột gây rò rỉ dữ liệu...")

# CHỈ XÓA 3 CỘT: Year (Thiên kiến lịch sử), Distance và Duration (Rò rỉ hậu quả)
cols_to_drop = ['Year', 'Distance(mi)', 'Duration']
df_sample = df_sample.drop(columns=[c for c in cols_to_drop if c in df_sample.columns], errors='ignore')

# Xóa các dòng khuyết thiếu dữ liệu để mô hình không bị lỗi toán học
df_sample = df_sample.dropna()

print(f"✅ Đã dọn dẹp xong! Kích thước dữ liệu sẵn sàng: {df_sample.shape}")
display(df_sample.head(3)) # In thử 3 dòng ra xem

In [ ]:
# ==========================================
# CELL 3: TRAIN/TEST SPLIT
# ==========================================
print("✂️ Đang chia tách dữ liệu Huấn luyện (Train) và Kiểm thử (Test)...")

X = df_sample.drop(columns=['Severity'])
y = df_sample['Severity']

# Chia tỷ lệ 80-20, dùng stratify để giữ nguyên tỷ lệ chênh lệch của các Mức độ tai nạn
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"📦 Tập Huấn luyện (Train) có: {X_train.shape[0]} dòng.")
print(f"📦 Tập Kiểm thử (Test) có: {X_test.shape[0]} dòng.")

In [ ]:
# ==========================================
# CELL 4: MÃ HÓA DỮ LIỆU NÂNG CAO (SMART ENCODING)
# Tối ưu hóa: Xử lý Lượng giác cho Thời gian & Tần suất cho Vị trí
# ==========================================
import numpy as np

print("🔢 Đang mã hóa dữ liệu (Biến chữ thành số & Xử lý tính chu kỳ)...")

# 1. NHÓM NHỊ PHÂN (True/False) -> Chuyển thành 1 và 0
bool_cols = ['Traffic_Signal', 'Junction'] 
for col in bool_cols:
    if col in X_train.columns:
        X_train[col] = X_train[col].astype(int)
        X_test[col] = X_test[col].astype(int)

# 2. NHÓM NGÀY/ĐÊM -> Chuyển thành 1 và 0
if 'Sunrise_Sunset' in X_train.columns:
    mapping = {'Day': 1, 'Night': 0}
    X_train['Sunrise_Sunset'] = X_train['Sunrise_Sunset'].map(mapping).fillna(0).astype(int)
    X_test['Sunrise_Sunset'] = X_test['Sunrise_Sunset'].map(mapping).fillna(0).astype(int)

# 3. KỸ THUẬT MỚI: CYCLICAL ENCODING CHO THỜI GIAN
print("🌀 Đang uốn cong thời gian bằng vòng tròn Lượng giác (Sin/Cos)...")
# Định nghĩa chu kỳ lớn nhất của từng mốc thời gian
time_cols = {'Hour': 24, 'Month': 12, 'Weekday': 7} 
for col, max_val in time_cols.items():
    if col in X_train.columns:
        # Tạo 2 cột Sin/Cos mới cho tập Train
        X_train[f'{col}_sin'] = np.sin(2 * np.pi * X_train[col] / max_val)
        X_train[f'{col}_cos'] = np.cos(2 * np.pi * X_train[col] / max_val)
        X_train = X_train.drop(columns=[col]) # Xóa cột gốc đi
        
        # Áp dụng công thức y hệt cho tập Test
        X_test[f'{col}_sin'] = np.sin(2 * np.pi * X_test[col] / max_val)
        X_test[f'{col}_cos'] = np.cos(2 * np.pi * X_test[col] / max_val)
        X_test = X_test.drop(columns=[col])

# 4. KỸ THUẬT FREQUENCY ENCODING CHO VỊ TRÍ & THỜI TIẾT
print("🏙️ Đang mã hóa Thành phố/Bang/Thời tiết theo Tần suất xuất hiện...")
# Với bài toán dự báo đa lớp (4 Mức độ Severity), dùng Tần suất xuất hiện là an toàn và ít tốn RAM nhất
freq_cols = ['State', 'City', 'Weather_Condition'] 
for col in freq_cols:
    if col in X_train.columns:
        # Dùng tập Train để đo lường
        freq = X_train[col].value_counts(normalize=True)
        
        # Áp dụng thước đo lên cả Train và Test
        X_train[col] = X_train[col].map(freq).fillna(0)
        X_test[col] = X_test[col].map(freq).fillna(0)

print("✅ CELL 4 HOÀN TẤT! Dữ liệu của bạn đã đạt chuẩn tối đa để đưa vào AI.")
display(X_train.head(3)) # In ra 3 dòng đầu để bạn kiểm tra kết quả biến hình

In [ ]:
# Kiểm tra xem có cột nào vô tình bị sót lại ở dạng chữ (object) không
print(X_train.info())

In [ ]:
# ==========================================
# CELL 5: HUẤN LUYỆN LIGHTGBM + CLASS WEIGHT & ĐÁNH GIÁ TOÀN DIỆN
# Tích hợp: Classification Report, F1-Score (Macro), ROC-AUC (OVR)
# ==========================================

import lightgbm as lgb
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, f1_score

print("🚀 Khởi động động cơ LightGBM (Chế độ: Phạt trọng số 'balanced')...")

# 1. KHỞI TẠO VÀ HUẤN LUYỆN
model_lgb = lgb.LGBMClassifier(
    n_estimators=100,          
    max_depth=-1,              
    learning_rate=0.1,         
    class_weight='balanced',   # Trọng số cân bằng 
    random_state=42,
    n_jobs=-1                  
)

print("🧠 Máy đang học trực tiếp từ dữ liệu gốc...")
model_lgb.fit(X_train, y_train)
print("✅ Học xong! Đang tính toán các chỉ số chuyên sâu...\n")

# 2. DỰ ĐOÁN KẾT QUẢ
# y_pred: Dùng để tính Accuracy, F1-Score và Classification Report
y_pred_lgb = model_lgb.predict(X_test)

# y_pred_proba: Dùng CỤ THỂ để tính ROC-AUC (Xác suất % rơi vào từng lớp)
y_pred_proba = model_lgb.predict_proba(X_test)

# 3. TÍNH TOÁN CÁC CHỈ SỐ ĐÁNH GIÁ
accuracy_lgb = accuracy_score(y_test, y_pred_lgb)

# F1-Score (Macro): Tính F1 cho từng lớp rồi chia trung bình cộng -> Công bằng cho Mức 4
f1_macro = f1_score(y_test, y_pred_lgb, average='macro')

# ROC-AUC (One-vs-Rest, Macro): Đánh giá khả năng phân tách của AI giữa các mức độ
roc_auc = roc_auc_score(y_test, y_pred_proba, multi_class='ovr', average='macro')

# 4. IN BẢNG THÀNH TÍCH (ĐỂ BÁO CÁO)
print("="*50)
print("🏆 BẢNG THÀNH TÍCH MÔ HÌNH LIGHTGBM (CLASS WEIGHT)")
print("="*50)
print(f"🎯 1. ĐỘ CHÍNH XÁC (Accuracy)  : {accuracy_lgb * 100:.2f}%")
print(f"⚖️ 2. ĐIỂM F1-MACRO            : {f1_macro:.4f}")
print(f"📈 3. ĐIỂM ROC-AUC (OVR-Macro) : {roc_auc:.4f}")
print("-" * 50)
print("📊 4. BÁO CÁO CHI TIẾT (CLASSIFICATION REPORT):")
print(classification_report(y_test, y_pred_lgb, zero_division=0))
print("="*50)